该文件从全省的住院病案首页数据中提取高血压患者数据，并进行数据清洗。

# 筛选高血压患者

In [ ]:
import pandas as pd
import numpy as np
import re
import os

In [ ]:
### 基本属性定义 ###

# 用到的列
usecols = [
    # 基本信息
    '医疗付款方式', '姓名', '性别', '出生日期', '年龄', '身份证号', 
    # 入院信息
    '入院时间', '出院时间', 
    # 费用
    '住院费', '自付金额',

    # 医院等级
    '医院等级(级)', '医院等级(等)',
    # '市', '区县',
    # 诊断信息
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
    ]

# 诊断名称
diag_cols = [
    '主要诊断', '其他诊断1', '其他诊断2', '其他诊断3', 
    '其他诊断4', '其他诊断5','其他诊断6', '其他诊断7', 
    '其他诊断8', '其他诊断9', '其他诊断10', '其他诊断11', 
    '其他诊断12', '其他诊断13', '其他诊断14', '其他诊断15'
    ]

# icd名称
icd_cols = [
    '主要诊断编码', '疾病编码2', '疾病编码3', 
    '疾病编码4', '疾病编码5', '疾病编码6', '疾病编码7', 
    '疾病编码8', '疾病编码9', '疾病编码10', '疾病编码11', 
    '疾病编码12', '疾病编码13', '疾病编码14', '疾病编码15'
]

In [ ]:
### 诊断编码规范化 - 只保留前三位 ###

# 预编译正则：字母+两位数字，可选.后一位字母或数字
icd_pattern = re.compile(r'^[A-Za-z]\d{2}$')

def normalize_icd_precise(code):
    
    # 如果是空值，返回空值
    if pd.isna(code):
        return np.nan
    
    # 保留到小数点后一位
    if "." in code:
        prefix, _ = code.split(".", 1)
    
    else:
        return np.nan

    # 用正则判断是否合法ICD编码
    if icd_pattern.match(prefix):
        return prefix.upper()
    else:
        return np.nan

In [ ]:
### 遍历数据文件筛选高血压患者并进行初步数据处理 ###

saved_list = []
hp_list = ['I10', 'I11', 'I12', 'I13', 'I14', 'I15']

path = r'E:\DataBase\全省his系统\行政区划'

# 性别映射表
gender_map = {
    '1.0': 1, '1': 1, '男': 1, 'M': 1,
    '2.0': 2, '2': 2, '女': 2, 'F': 2,
    0: np.nan, 9: np.nan
}

for d in os.listdir(path):
    if 'nan' in d:
        continue

    df_list = []

    # 遍历一个城市下面的所有县区目录
    for root, subdirs, files in os.walk(os.path.join(path, d)):
        city_name = root.split('\\')[-2]
        county_name = root.split('\\')[-1]

        if city_name in saved_list:
            continue

        for file in files:
            if not file.endswith('.csv'):
                continue
            
            if file.split('_')[0] == '2025':
                continue

            file_path = os.path.join(root, file)
            df = pd.read_csv(file_path, usecols=usecols)

            # 添加市、县区信息
            df['市'] = city_name
            df['区县'] = county_name

            # ICD 处理
            df[icd_cols] = df[icd_cols].astype(str).applymap(normalize_icd_precise)

            # 只保留高血压患者的记录
            mask = df[icd_cols].isin(hp_list)
            df = df[mask.any(axis=1)].copy()

            df_list.append(df)

    if not df_list:  # 没有数据就跳过
        continue

    df = pd.concat(df_list, ignore_index=True)
    df = df[df.columns].astype(str)

    # 输出路径
    path_export = os.path.join(r'E:\DataBase\全省his系统', 'hp-citylevel-R1', f'{city_name}.parquet')

    # 保存 parquet
    df.to_parquet(path_export, engine='pyarrow', compression='snappy')

In [ ]:
### 合并文件 ###

folder_path = r"E:\DataBase\全省his系统\hp-citylevel-R1"

# 获取所有parquet文件路径
parquet_files = [
    os.path.join(folder_path, f) 
    for f in os.listdir(folder_path) 
    if f.endswith('.parquet')
]

# 一次性读取所有文件并合并
df = pd.concat(
    [pd.read_parquet(f) for f in parquet_files],
    ignore_index=True
)

print("所有诊断中有高血压的就诊记录数:", df.shape[0])

所有诊断中有高血压的就诊记录数: 14429897


# 数据清洗

In [ ]:
### 去重 ###
df = df.drop_duplicates(subset=['身份证号', '入院时间'])
print('去重后有', df.shape[0],'条入院记录')

去重后有 8267735 条入院记录


In [ ]:
### 医疗付款方式规范化 ###
payment_regulaztion_map = {
    '2.0': '2',
    '3.0': '3',
    '1.0': '1',
    '7.0': '7',
    '99.0': '99',
    '9.0': '9',
    '4.0': '4',
    '6.0': '6',
    '8.0': '8',
    '5.0': '5',
    '2': '2',
    '3': '3',
    '1': '1',
    '7': '7',
    'nan': pd.NA,
    '9': '9',
    '11.0': pd.NA,
    '02': '2',
    '03': '3',
    '4': 4,
    '10.0': pd.NA,
    '6':'6',
    '99': '99',
    '01':'1',
    '07': '7',
    '2.1': pd.NA,
    '8':'8',
    '04':'4',
    '06':'6',
    '0.0':pd.NA,
    '1.1':pd.NA,
    '3.1':pd.NA,
    '09':'9',
    '5':'5',
    '05':'5',
    '22.0':pd.NA,
    '0-':pd.NA
}
# 转化类别
pay_map = {
    '1': "UEBMI",      # 城镇职工基本医疗保险
    '2': "URRBMI",   # 城镇居民基本医疗保险
    '3': "URRBMI",   # 新型农村合作医疗
    '4': "GS", # 贫困救助
    '5': "COI", # 商业医疗保险
    '6': "GS", # 全公费
    '7': "SP",       # 全自费
    '8': "COI", # 其他社会保险
    '9': "COI", # 其他
    '99': "COI" # 其他
}

df['医疗付款方式'] = df['医疗付款方式'].map(payment_regulaztion_map)
df['医疗付款方式'] = df['医疗付款方式'].map(pay_map)

df['医疗付款方式'].value_counts()

医疗付款方式
URRBMI    4768436
UEBMI     2084934
COI        708645
SP         584046
GS         119553
Name: count, dtype: int64

In [ ]:
### 性别规范化 ###
# df['性别'] = df['性别'].replace(gender_map)
gender_map = {
    '1.0': 1, '1': 1, '男': 1, 'M': 1,
    '2.0': 0, '2': 0, '女': 0, 'F': 0,
    '0': np.nan, '9': np.nan,
    '0.0': np.nan, '9.0': np.nan,
    'nan': np.nan
}

df['性别'] = df['性别'].map(gender_map)
df['性别'].value_counts()

性别
1.0    4264826
0.0    3991567
Name: count, dtype: int64

In [ ]:
### 年龄规范化 ###
df['年龄'] = pd.to_numeric(df['年龄'], errors='coerce')
df['年龄'] = np.where(df['年龄'] > 150, pd.NA, df['年龄'])

In [ ]:
### 身份证号规范化 ###
pattern = r'^`[1-9]\d{5}(?:18|19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\d{3}[\dXx]$'

def check_id_format(s):
    if pd.isna(s):
        return np.nan
    if bool(re.match(pattern, s)):
        return s
    else:
        return np.nan
df['身份证号'] = df['身份证号'].map(check_id_format)

In [ ]:
### 入院时间规范化 ###
def clean_date_column(df, colname):
    """
    清洗日期列，统一格式为 YYYY-MM-DD
    - 支持斜杠/横杠
    - 自动补零（单个数字前补0）
    - 去除时分秒
    - 解析失败的值转换为 NaT
    """
    # 1. 转为字符串并清理
    df[colname] = (
        df[colname]
        .astype(str)
        .str.strip()
        .str.replace('/', '-', regex=False)                     # 统一分隔符
        .str.replace(r'(\D)(\d)(?=\D)', r'\g<1>0\2', regex=True)  # 单位数补零
        .str.slice(0, 10)                                       # 保留前10位 (YYYY-MM-DD)
    )

    # 2. 转为日期
    df[colname] = pd.to_datetime(df[colname], errors='coerce').dt.normalize()


    return df  

df = clean_date_column(df, '入院时间')

# 将入院时间在2019-2024年之外的入院时间记作缺失
df.loc[(df['入院时间'].dt.year > 2024) | (df['入院时间'].dt.year < 2019), '入院时间'] = pd.NA

In [ ]:
### 住院费规范化 ###

df['住院费'] = np.where(df['住院费'] == 'nan', pd.NA, df['住院费'])
df['住院费'] = pd.to_numeric(df['住院费'])
(df['住院费'] <= 0).sum()

# 将住院费用小于等于0的处理为缺失值
df.loc[df['住院费'] <= 0, '住院费'] = pd.NA

np.int64(29036)

In [13]:
# 自付金额规范化
df['自付金额'] = np.where(df['自付金额'] == 'nan', pd.NA, df['自付金额'])
df['自付金额'] = pd.to_numeric(df['自付金额'])
(df['自付金额'] < 0).sum()

np.int64(7928)

In [14]:
# 将自付金额用小于等于0的处理为缺失值
df.loc[df['自付金额'] < 0, '自付金额'] = pd.NA

In [15]:
print('身份证号', df['身份证号'].isna().sum(), '个')
print('医疗付款方式缺失为', df['医疗付款方式'].isna().sum(), '个')
print('入院时间', df['入院时间'].isna().sum(), '个')
print('性别缺失为', df['性别'].isna().sum(), '个')
print('年龄缺失为', df['年龄'].isna().sum(), '个')
print('住院费', df['住院费'].isna().sum(), '个')
print('自付金额', df['自付金额'].isna().sum(), '个')

身份证号 8881 个
医疗付款方式缺失为 2121 个
入院时间 90527 个
性别缺失为 11342 个
年龄缺失为 6 个
住院费 29414 个
自付金额 127094 个


In [16]:
# print('含有关键变量缺失的就诊记录总数为：', df[['身份证号', '医疗付款方式', '入院时间', '性别', '年龄']].isna().any(axis=1).sum(), '条')
print('含有关键变量缺失的就诊记录总数为：', df[['身份证号', '医疗付款方式', '入院时间', '性别', '年龄', '住院费', '自付金额']].isna().any(axis=1).sum(), '条')

含有关键变量缺失的就诊记录总数为： 263696 条


In [ ]:
# 提取存在缺失的住院数据
df_na = df[df[['身份证号', '医疗付款方式', '入院时间', '性别', '年龄', '住院费', '自付金额']].isna().any(axis=1)]
# 提取用于MI的数据
df_MI = df.dropna(subset=['身份证号', '医疗付款方式'])

In [18]:
# df = df.dropna(subset=['身份证号', '医疗付款方式', '入院时间', '性别', '年龄'])
df = df.dropna(subset=['身份证号', '医疗付款方式', '入院时间', '性别', '年龄', '住院费', '自付金额'])
print('去除缺失值后，数据量为：', df.shape[0])

去除缺失值后，数据量为： 8004039


In [ ]:
# 保存数据
# df.to_parquet(r"File/hpconcat.parquet")
# df_na.to_parquet(r"File/hpconcat_na.parquet")
# df_MI.to_parquet(r"File/hpconcat_mi.parquet")